# Notebook 02 — Feature Engineering
## JFK Airport Daily Passenger Throughput Forecasting


**Design decisions** are documented in `feature_engineering_decisions.md`.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json
from datetime import date, timedelta
import holidays

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

FIGURES = '../reports/figures'
os.makedirs(FIGURES, exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. Load Merged Dataset from Notebook 01

In [2]:
df = pd.read_csv('../data/processed/jfk_daily_merged.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Columns ({df.shape[1]}): {list(df.columns)}')
df.head(3)

Shape: (2337, 30)
Date range: 2018-12-30 to 2025-05-31
Columns (30): ['Date', 'daily_throughput', 'JFK Terminal 1', 'JFK Terminal 2', 'JFK Terminal 4 Delta 1', 'JFK Terminal 4 FIS CP', 'JFK Terminal 4 Main', 'JFK Terminal 5', 'JFK Terminal 7', 'JFK Terminal 8', 'AWND', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'WSF2', 'WSF5', 'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT08', 'WT09', 'scheduled_departures', 'scheduled_arrivals', 'total_scheduled_flights']


,Date,daily_throughput,JFK Terminal 1,JFK Terminal 2,JFK Terminal 4 Delta 1,JFK Terminal 4 FIS CP,JFK Terminal 4 Main,JFK Terminal 5,JFK Terminal 7,JFK Terminal 8,AWND,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WSF2,WSF5,WT01,WT02,WT03,WT04,WT05,WT06,WT08,WT09,scheduled_departures,scheduled_arrivals,total_scheduled_flights
0,2018-12-30,"98,287.0000","12,170.0000","7,945.0000",0.0000,918.0000,"31,154.0000","24,076.0000","7,978.0000","14,046.0000",5.3700,0.0000,0.0000,0.0000,37.0000,39.0000,32,19.9000,23.0000,0,0,0,0,0,0,0,0,NaN,NaN,NaN
1,2018-12-31,"79,117.0000","10,481.0000","5,035.0000",0.0000,633.0000,"23,919.0000","21,330.0000","6,202.0000","11,517.0000",9.1700,1.2300,0.0000,0.0000,37.0000,49.0000,30,21.9000,28.0000,1,1,0,0,0,0,0,0,NaN,NaN,NaN
2,2019-01-01,"89,947.0000","12,193.0000","6,051.0000",0.0000,993.0000,"26,152.0000","22,099.0000","8,002.0000","14,457.0000",18.3400,0.1100,0.0000,0.0000,51.0000,59.0000,40,33.1000,40.9000,1,1,0,0,0,0,0,0,325.0000,320.0000,645.0000


## 3. Drop Unnecessary Columns

Terminal-level columns were used for EDA only — the target `daily_throughput`
is already the sum. We also drop `scheduled_arrivals` and
`total_scheduled_flights` per
[Decision #1](feature_engineering_decisions.md#1-scheduled-departures-only-not-arrivals):
TSA screens only departing passengers, and departures/arrivals are r ≈ 0.99
correlated.

In [3]:
# Drop terminal columns (EDA only — target is already their sum)
terminal_cols = [c for c in df.columns if 'Terminal' in c]
df = df.drop(columns=terminal_cols)
print(f'Dropped {len(terminal_cols)} terminal columns: {terminal_cols}')

# Drop arrivals and total flights (Decision #1 — departures only)
flight_drop = ['scheduled_arrivals', 'total_scheduled_flights']
flight_drop = [c for c in flight_drop if c in df.columns]
df = df.drop(columns=flight_drop)
print(f'Dropped flight columns: {flight_drop}')

print(f'\nRemaining: {df.shape[1]} columns: {list(df.columns)}')

Dropped 8 terminal columns: ['JFK Terminal 1', 'JFK Terminal 2', 'JFK Terminal 4 Delta 1', 'JFK Terminal 4 FIS CP', 'JFK Terminal 4 Main', 'JFK Terminal 5', 'JFK Terminal 7', 'JFK Terminal 8']
Dropped flight columns: ['scheduled_arrivals', 'total_scheduled_flights']

Remaining: 20 columns: ['Date', 'daily_throughput', 'AWND', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'WSF2', 'WSF5', 'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT08', 'WT09', 'scheduled_departures']


## 4. Calendar Features (Cyclical Encoding)

Per [Decision #5](feature_engineering_decisions.md#5-cyclical-encoding-for-calendar-features),
we encode `day_of_week`, `month`, and `day_of_year` with sin/cos transforms
to preserve circularity (Sunday↔Monday, December↔January).

We also add `is_weekend` as a binary feature. `quarter` is dropped per
[Decision #6](feature_engineering_decisions.md#6-drop-quarter-redundant-with-month-encoding)
— it is redundant with the finer-grained month encoding.

In [4]:
# Extract raw calendar values
df['day_of_week'] = df['Date'].dt.dayofweek      # 0=Mon, 6=Sun
df['month']       = df['Date'].dt.month           # 1-12
df['day_of_year'] = df['Date'].dt.dayofyear       # 1-366
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# Cyclical sin/cos encoding
def cyclical_encode(df, col, period):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
    return df

df = cyclical_encode(df, 'day_of_week', 7)
df = cyclical_encode(df, 'month', 12)

# day_of_year: use actual year length (365 or 366) for correct alignment
days_in_year = df['Date'].dt.is_leap_year.map({True: 366, False: 365})
df = cyclical_encode(df, 'day_of_year', days_in_year)

# Drop raw integer versions (keep only sin/cos + is_weekend)
# quarter is dropped entirely — redundant with month encoding (Decision #6)
df = df.drop(columns=['day_of_week', 'month', 'day_of_year'])

calendar_features = ['is_weekend',
                     'day_of_week_sin', 'day_of_week_cos',
                     'month_sin', 'month_cos',
                     'day_of_year_sin', 'day_of_year_cos']
print(f'Calendar features ({len(calendar_features)}): {calendar_features}')

Calendar features (7): ['is_weekend', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']


## 5. Holiday Features

We use the `holidays` library to generate US federal holidays, which correctly
handles **observed dates** (e.g., when July 4th falls on Saturday, Friday is
the observed holiday — the day that actually shifts airport demand). We then
add travel-period flags for major holiday seasons.

In [5]:
# ── US Federal Holidays via holidays library ──
# Handles observed dates automatically (e.g., Jul 4 on Sat → Fri observed)
years = range(df['Date'].dt.year.min(), df['Date'].dt.year.max() + 1)
us_hols = holidays.US(years=years)

holiday_dates = set(us_hols.keys())

# ── Basic holiday flags ──
df['is_holiday'] = df['Date'].dt.date.isin(holiday_dates).astype(int)

# is_long_weekend: is this day part of a 3-day weekend created by a Mon/Fri holiday?
def is_long_weekend_day(dt):
    d = dt.date()
    for hol_date in holiday_dates:
        dow = hol_date.weekday()
        if dow == 0:  # Monday holiday → Sat/Sun/Mon
            if hol_date - timedelta(days=2) <= d <= hol_date:
                return 1
        elif dow == 4:  # Friday holiday → Fri/Sat/Sun
            if hol_date <= d <= hol_date + timedelta(days=2):
                return 1
    return 0

df['is_long_weekend'] = df['Date'].apply(is_long_weekend_day)

print(f'Federal holidays in range: {len(us_hols)}')
print(f'Holiday days in dataset: {df["is_holiday"].sum()}')
print(f'Long weekend days in dataset: {df["is_long_weekend"].sum()}')
print(f'\nSample holidays:')
for d, name in sorted(us_hols.items())[:10]:
    print(f'  {d}: {name}')

Federal holidays in range: 95
Holiday days in dataset: 77
Long weekend days in dataset: 152

Sample holidays:
  2018-01-01: New Year's Day
  2018-01-15: Martin Luther King Jr. Day
  2018-02-19: Washington's Birthday
  2018-05-28: Memorial Day
  2018-07-04: Independence Day
  2018-09-03: Labor Day
  2018-10-08: Columbus Day
  2018-11-11: Veterans Day
  2018-11-12: Veterans Day (observed)
  2018-11-22: Thanksgiving Day


### 5.1 Holiday Travel Period Flags

Airport demand surges don't just happen on the holiday itself — they span
multi-day travel windows. We flag the peak travel periods around major
holidays.

In [6]:
def thanksgiving_period(dt):
    """Tue before Thanksgiving through Sun after (6 days)."""
    y = dt.year
    # Find Thanksgiving: 4th Thursday in November
    d = date(y, 11, 1)
    d += timedelta(days=(3 - d.weekday()) % 7)  # first Thursday
    tg = d + timedelta(weeks=3)
    start = tg - timedelta(days=2)   # Tuesday before
    end = tg + timedelta(days=3)     # Sunday after
    return int(start <= dt.date() <= end)

def christmas_period(dt):
    """Dec 20 – Jan 2 (captures both Christmas and New Year travel)."""
    d = dt.date()
    if d.month == 12 and d.day >= 20:
        return 1
    if d.month == 1 and d.day <= 2:
        return 1
    return 0

def july4_period(dt):
    """Jul 1 – Jul 7 (week around Independence Day)."""
    d = dt.date()
    return int(d.month == 7 and 1 <= d.day <= 7)

def spring_break_period(dt):
    """Mid-March to mid-April (broad spring break window)."""
    d = dt.date()
    return int((d.month == 3 and d.day >= 15) or (d.month == 4 and d.day <= 15))

df['period_thanksgiving'] = df['Date'].apply(thanksgiving_period)
df['period_christmas']    = df['Date'].apply(christmas_period)
df['period_july4']        = df['Date'].apply(july4_period)
df['period_spring_break'] = df['Date'].apply(spring_break_period)

holiday_features = ['is_holiday', 'is_long_weekend',
                    'period_thanksgiving', 'period_christmas',
                    'period_july4', 'period_spring_break']
print(f'Holiday features ({len(holiday_features)}): {holiday_features}')
for f in holiday_features:
    print(f'  {f}: {df[f].sum()} days flagged')

Holiday features (6): ['is_holiday', 'is_long_weekend', 'period_thanksgiving', 'period_christmas', 'period_july4', 'period_spring_break']
  is_holiday: 77 days flagged
  is_long_weekend: 152 days flagged
  period_thanksgiving: 36 days flagged
  period_christmas: 88 days flagged
  period_july4: 41 days flagged
  period_spring_break: 224 days flagged


## 6. Autoregressive Features (Lags)

Lags provide the model with recent throughput history. We use lags of 1, 7,
14, and 28 days. The year-over-year signal is captured separately in §7.

In [7]:
# Sort by date to ensure correct lag computation
df = df.sort_values('Date').reset_index(drop=True)

lag_days = [1, 7, 14, 28]
for lag in lag_days:
    df[f'lag_{lag}'] = df['daily_throughput'].shift(lag)

lag_features = [f'lag_{d}' for d in lag_days]
print(f'Lag features ({len(lag_features)}): {lag_features}')
print(f'NaN introduced: lag_28 creates {df["lag_28"].isna().sum()} NaN rows')

Lag features (4): ['lag_1', 'lag_7', 'lag_14', 'lag_28']
NaN introduced: lag_28 creates 28 NaN rows


## 7. Same-Weekday-Last-Year

Per [Decision #3](feature_engineering_decisions.md#3-same-weekday-last-year-not-lag_365),
we compute the mean throughput of the same weekday in the same month of the
prior year. This is more robust than raw `lag_365` (averaged over ~4 days
instead of a single observation) and correctly aligns with day-of-week
patterns.

**Cost:** ~365 rows lost to NaN (first year has no prior-year reference).

In [8]:
def same_weekday_last_year(df):
    """For each row, average throughput of same weekday in same month, prior year."""
    df = df.copy()
    df['_year'] = df['Date'].dt.year
    df['_month'] = df['Date'].dt.month
    df['_dow'] = df['Date'].dt.dayofweek

    # Build lookup: (year, month, dow) -> mean throughput
    lookup = (df.groupby(['_year', '_month', '_dow'])['daily_throughput']
              .mean()
              .to_dict())

    # For each row, look up (year-1, month, dow)
    df['same_weekday_last_year'] = df.apply(
        lambda r: lookup.get((r['_year'] - 1, r['_month'], r['_dow']), np.nan),
        axis=1
    )

    df = df.drop(columns=['_year', '_month', '_dow'])
    return df

df = same_weekday_last_year(df)

n_nan = df['same_weekday_last_year'].isna().sum()
print(f'same_weekday_last_year: {n_nan} NaN rows (first year without prior reference)')
print(f'Sample values (non-NaN):')
print(df[df['same_weekday_last_year'].notna()][['Date', 'daily_throughput', 'same_weekday_last_year']].head())

same_weekday_last_year: 357 NaN rows (first year without prior reference)
Sample values (non-NaN):
          Date  daily_throughput  same_weekday_last_year
336 2019-12-01       97,860.0000             98,287.0000
337 2019-12-02       88,745.0000             79,117.0000
343 2019-12-08       95,095.0000             98,287.0000
344 2019-12-09       88,963.0000             79,117.0000
350 2019-12-15       95,727.0000             98,287.0000


## 8. Rolling Statistics

Rolling windows capture recent trend and volatility. We compute mean, std,
min, and max over 7-, 14-, and 30-day windows.

**Important:** We use `.shift(1)` before rolling to avoid target leakage — the
rolling window must not include the current day's throughput.

In [9]:
# Shift throughput by 1 to prevent leakage (window ends at day t-1)
shifted = df['daily_throughput'].shift(1)

windows = [7, 14, 30]
for w in windows:
    roll = shifted.rolling(window=w, min_periods=w)
    df[f'roll_mean_{w}']  = roll.mean()
    df[f'roll_std_{w}']   = roll.std()
    df[f'roll_min_{w}']   = roll.min()
    df[f'roll_max_{w}']   = roll.max()

rolling_features = [f'roll_{stat}_{w}' for w in windows for stat in ['mean', 'std', 'min', 'max']]
print(f'Rolling features ({len(rolling_features)}): {rolling_features}')
print(f'NaN from roll_mean_30: {df["roll_mean_30"].isna().sum()} rows')

Rolling features (12): ['roll_mean_7', 'roll_std_7', 'roll_min_7', 'roll_max_7', 'roll_mean_14', 'roll_std_14', 'roll_min_14', 'roll_max_14', 'roll_mean_30', 'roll_std_30', 'roll_min_30', 'roll_max_30']
NaN from roll_mean_30: 30 rows


## 9. Differencing Features

Differencing captures momentum and trend changes at different time scales.

In [10]:
# Day-over-day change
df['diff_1'] = df['daily_throughput'].diff(1)

# Week-over-week change (same day last week)
df['diff_7'] = df['daily_throughput'].diff(7)

# Year-over-year change (364 days to stay aligned on same weekday)
# df['diff_364'] = df['daily_throughput'].diff(364)

diff_features = ['diff_1', 'diff_7'] 
print(f'Differencing features ({len(diff_features)}): {diff_features}')

Differencing features (2): ['diff_1', 'diff_7']


## 10. COVID Dummy Variables

Per [Decision #2](feature_engineering_decisions.md#2-three-level-covid-encoding-not-continuous-curve):
three-level encoding with normal operations as baseline.

- `covid_acute`: Mar 15, 2020 – Jun 30, 2020 (lockdowns, near-zero traffic)
- `covid_recovery`: Jul 1, 2020 – Jun 30, 2022 (gradual return)
- Baseline (both = 0): Normal operations

In [11]:
df['covid_acute'] = ((df['Date'] >= '2020-03-15') &
                     (df['Date'] <= '2020-06-30')).astype(int)

df['covid_recovery'] = ((df['Date'] >= '2020-07-01') &
                        (df['Date'] <= '2022-06-30')).astype(int)

covid_features = ['covid_acute', 'covid_recovery']
print(f'COVID features: {covid_features}')
print(f'  covid_acute days:    {df["covid_acute"].sum()}')
print(f'  covid_recovery days: {df["covid_recovery"].sum()}')
print(f'  normal (baseline):   {(~(df["covid_acute"] | df["covid_recovery"]).astype(bool)).sum()}')

COVID features: ['covid_acute', 'covid_recovery']
  covid_acute days:    108
  covid_recovery days: 730
  normal (baseline):   1499
